# S1–S6 resolution convergence: G_max vs azimuthal cells (Fig 7, Table 4, Fig 6)

Maximum normalised density-amplitude growth at ~320 km as a function of the number of azimuthal cells $N_\phi$, with the empirical fit of Eq. 10, $G_{max}(N_\phi) = A - B/N_\phi$, and the 1D Cartesian (T23) and radial (P26) references.

* **G_max** for a run is the max over its forced columns and altitudes of the **last-cycle** growth peak (`growth_peaks[:, -1]`), which is the manuscript definition.
* **Sections (a)–(c)** use the **staging caches** with `STAGING_RECIPE`:
  * The background is averaged over all azimuth columns and the 17 pre-forcing outputs that every run has (t = 4908–5012 s). Averaging S4–S6's whole pre-forcing record would include the spin-up, about +2.9% at 315 km.
  * The growth window runs over the cells nearest 150 and 320 km, i.e. 148.3–315.4 km, the same window as the 2024 pipeline.
  * The root `optimized_loader` background (`mad_column0`) uses azimuth column 0 alone. That leaves particle noise of several % in each altitude's reference, which is what made the S1–S3 profiles crooked.
  * The fit band is Student-t with dof = n − 2.
* **Section (d)** reproduces the published Fig 7 / Table 4 / Fig 6 from the frozen 2024 parquet (`zenodo_release/2d_growth_peak_data.parquet`) and overlays it with the staging results.
* **Method:** `dsmclib.resolution`. Checks: `checks/regress_resolution_fit.py` (published numbers) and `checks/regress_legacy_2024_growth.py` (2024 pkls from raw data).

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

from dsmclib import plotting
from dsmclib import resolution as R
from dsmclib.constants import LEGACY_RADIUS_KM, LEGACY_T23, T23_GROWTH_MAX, TIAN_ALTITUDES, TIAN_GROW
from dsmclib.paths import LEGACY_ZENODO, output_dir

plotting.use_style(font_scale=1.4)
FIG_DIR = output_dir(R.CAMPAIGN, "figures")
TAB_DIR = output_dir(R.CAMPAIGN, "tables")

## (a) Fig 7 from the staging caches

In [2]:
profiles = R.last_cycle_profiles()          # S1..S6 (S4 = 2024 run); reads caches only
gmax = R.gmax_table(profiles)
popt, pcov = R.fit_inverse_n(gmax["N_phi"], gmax["G_max"])
fit = R.fit_summary(gmax, popt, pcov, dist="t")
p26 = R.p26_reference()
display(gmax)
pd.Series(fit)

,Simulation,N_phi,G_max,alt_at_max_km,azi_at_max,n_columns
0,S1,128,5.251900,315.355157,67azi,12
1,S2,256,5.596256,315.355157,137azi,23
2,S3,384,5.728726,315.355157,219azi,34
3,S4,512,5.701512,315.355157,292azi,44
4,S5,640,5.902473,315.355157,350azi,55
5,S6,1024,5.725779,315.355157,556azi,88


A                5.9084
A_err          0.059151
A_lo            5.74417
A_hi           6.072631
B             82.048757
B_err         15.248655
B_lo          39.711703
B_hi         124.385811
corr_AB        0.808392
rms_resid      0.069638
r2             0.878612
level              0.95
dist                  t
quantile       2.776445
n_points              6
dtype: object

In [3]:
references = {"1D Cartesian (T23)": T23_GROWTH_MAX, "1D Radial (P26)": p26}

fig, ax = plt.subplots(figsize=(6.5, 4.5))
plotting.plot_resolution_fit(ax, gmax, popt, pcov, dist="t", references=references)
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / "resolution_fit_staging.png", dpi=300, bbox_inches="tight")

In [4]:
# Same data in the manuscript's Figure 7 style (rev4); this file goes into the docx
fig = plotting.plot_resolution_fit_manuscript(gmax, popt, pcov, t23=T23_GROWTH_MAX, p26=p26, dist="t")
fig.savefig(FIG_DIR / "Figure_7_manuscript.png", dpi=260)

## (b) Table 4 from the staging caches

In [5]:
table4 = R.resolution_table(gmax, popt)
table4.to_csv(TAB_DIR / "table4_resolution_staging.csv", index=False, float_format="%.4f")
print(f"A = {fit['A']:.3f} ± {fit['A_err']:.3f}, B = {fit['B']:.1f} ± {fit['B_err']:.1f}, R² = {fit['r2']:.3f}")
table4

A = 5.908 ± 0.059, B = 82.0 ± 15.2, R² = 0.879


,Simulation,N_phi,G_max,alt_at_max_km,azi_at_max,n_columns,cell_width_km,cells_per_lambda_h,offset_pct
0,S1,128,5.251900,315.355157,67azi,12,171.290431,2.706514,-11.111305
1,S2,256,5.596256,315.355157,137azi,23,85.645215,5.413029,-5.283055
2,S3,384,5.728726,315.355157,219azi,34,57.096810,8.119543,-3.041003
3,S4,512,5.701512,315.355157,292azi,44,42.822608,10.826057,-3.501594
4,S5,640,5.902473,315.355157,350azi,55,34.258086,13.532571,-0.100317
5,S6,1024,5.725779,315.355157,556azi,88,21.411304,21.652114,-3.090880


## (c) Fig 6 from the staging caches

The P26 reference is drawn once, from the 1D_5p cache. The published figure drew the same data twice, as raw "Spherical 1D" and savgol-smoothed "Harrah 1D". T23 is the full digitised Tian et al. (2023) curve from `dsmclib.constants`.

In [6]:
p26_alt, p26_growth = R.p26_profile()
refs = [
    dict(x=TIAN_GROW, y=TIAN_ALTITUDES, label="1D Cartesian (T23)", color="k"),
    dict(x=p26_growth, y=p26_alt, label="1D Radial (P26)", color="#2ca02c"),
]
fig = plotting.plot_growth_profile_facets(R.add_distance_from_centre(profiles), refs)
fig.savefig(FIG_DIR / "growth_profiles_S1-S6.png", dpi=300, bbox_inches="tight")

## (d) Legacy: the published numbers (frozen 2024 parquet)

`2d_growth_peak_data.parquet` is byte-identical to `ArchiveSims/growth_peak_data.parquet` (2024-07-25). It aggregates the 2024 run-folder `growth_peak_data_<col>azi_*.pkl` caches, whose generator is lost. `checks/regress_legacy_2024_growth.py` reconstructs their recipe from raw data:
* the `azimean_snapshot` background (a single snapshot, 2 outputs before forcing, averaged over all columns),
* a time window from forcing start to forcing end + 600 s,
* the cells **nearest** 150 and 320 km (148.3–315.4 km),
* the unchanged peak detection.

That reproduces S2 and S3 bit for bit, S1 to 1e-14, and S6 on 87/88 columns. S6 column 512 does not reproduce even from its own stored series, so it was picked by hand. S4 and S5 have no 2024 pkls on disk, and S5 has 56 columns in the parquet against 55 forced columns in staging.

The staging recipe differs from 2024 only in averaging the background over 17 snapshots instead of 1, and in 700 s of padding instead of 600 (which makes no difference).

This section reproduces `zenodo_release/Figure_6_and_7.py` exactly: normal-z band, the T23 line = 6.14065, and cell width at 3390 + 100 km.

In [7]:
legacy = R.legacy_profiles()
gmax_legacy = R.gmax_table(legacy)
popt_legacy, pcov_legacy = R.fit_inverse_n(gmax_legacy["N_phi"], gmax_legacy["G_max"])
fit_legacy = R.fit_summary(gmax_legacy, popt_legacy, pcov_legacy, dist="normal")
table4_legacy = R.resolution_table(gmax_legacy, popt_legacy, radius_km=LEGACY_RADIUS_KM)
table4_legacy.to_csv(TAB_DIR / "table4_resolution_legacy.csv", index=False, float_format="%.4f")
print(f"A = {fit_legacy['A']:.3f} ± {fit_legacy['A_err']:.3f}, B = {fit_legacy['B']:.1f} ± {fit_legacy['B_err']:.1f}, R² = {fit_legacy['r2']:.3f}")
table4_legacy

A = 5.832 ± 0.036, B = 67.1 ± 9.2, R² = 0.929


,Simulation,N_phi,G_max,alt_at_max_km,azi_at_max,n_columns,cell_width_km,cells_per_lambda_h,offset_pct
0,S1,128,5.284117,315.355157,67azi,12,171.314974,2.706127,-9.395333
1,S2,256,5.595048,315.355157,137azi,23,85.657487,5.412253,-4.063919
2,S3,384,5.688399,315.355157,219azi,34,57.104991,8.118380,-2.463277
3,S4,512,5.763593,315.355157,284azi,44,42.828744,10.824506,-1.173941
4,S5,640,5.685489,315.355157,350azi,56,34.262995,13.530633,-2.513160
5,S6,1024,5.712851,315.355157,556azi,88,21.414372,21.649012,-2.044006


In [8]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
plotting.plot_resolution_fit(ax, gmax_legacy, popt_legacy, pcov_legacy, dist="normal",
                             references={"1D Cartesian (T23)": LEGACY_T23, "1D Spherical (P26)": p26})
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / "resolution_fit_legacy.png", dpi=300, bbox_inches="tight")

### Legacy vs staging on one axis

* Grey: the published 2024 numbers.
* Red/blue: staging with `STAGING_RECIPE` (sections a–c).
* Orange: staging with the root recipe (`mad_column0`, first cell above 150 / 320 km, i.e. 153.5–327.4 km), kept for comparison.

In [9]:
gmax_col0 = R.gmax_table(R.last_cycle_profiles(background="mad_column0", altitude_cells="first_above"))
popt_col0, pcov_col0 = R.fit_inverse_n(gmax_col0["N_phi"], gmax_col0["G_max"])
fit_col0 = R.fit_summary(gmax_col0, popt_col0, pcov_col0, dist="t")

fig, ax = plt.subplots(figsize=(8, 5.5))
plotting.plot_resolution_fit(ax, gmax_legacy, popt_legacy, pcov_legacy, dist="normal", color="0.5",
                             point_color="0.4", label="2024 parquet",
                             references={"1D Cartesian (T23)": T23_GROWTH_MAX, "1D Radial (P26)": p26})
plotting.plot_resolution_fit(ax, gmax_col0, popt_col0, pcov_col0, dist="t", color="orange", point_color="orange",
                             label="staging, root recipe")
plotting.plot_resolution_fit(ax, gmax, popt, pcov, dist="t", color="r", point_color="C0", label="staging")
ax.legend(fontsize=7, loc="lower right", ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "resolution_fit_legacy_vs_staging.png", dpi=300, bbox_inches="tight")

In [10]:
diff = gmax_legacy.merge(gmax, on=["Simulation", "N_phi"], suffixes=("_legacy", "_staging"))
diff["G_max_root_recipe"] = gmax_col0["G_max"].to_numpy()
diff["G_max_diff"] = diff["G_max_staging"] - diff["G_max_legacy"]
diff[["Simulation", "N_phi", "G_max_legacy", "G_max_staging", "G_max_root_recipe", "G_max_diff",
      "alt_at_max_km_legacy", "alt_at_max_km_staging", "azi_at_max_legacy", "azi_at_max_staging",
      "n_columns_legacy", "n_columns_staging"]]

,Simulation,N_phi,G_max_legacy,G_max_staging,G_max_root_recipe,G_max_diff,alt_at_max_km_legacy,alt_at_max_km_staging,azi_at_max_legacy,azi_at_max_staging,n_columns_legacy,n_columns_staging
0,S1,128,5.284117,5.251900,4.414066,-0.032217,315.355157,315.355157,67azi,67azi,12,12
1,S2,256,5.595048,5.596256,6.467261,0.001208,315.355157,315.355157,137azi,137azi,23,23
2,S3,384,5.688399,5.728726,4.999475,0.040327,315.355157,315.355157,219azi,219azi,34,34
3,S4,512,5.763593,5.701512,5.473323,-0.062081,315.355157,315.355157,284azi,292azi,44,44
4,S5,640,5.685489,5.902473,5.959000,0.216984,315.355157,315.355157,350azi,350azi,56,55
5,S6,1024,5.712851,5.725779,6.055942,0.012928,315.355157,315.355157,556azi,556azi,88,88


In [11]:
pd.DataFrame({"legacy (normal z)": fit_legacy, "staging (t)": fit, "staging, root recipe (t)": fit_col0}).T[
    ["A", "A_err", "A_lo", "A_hi", "B", "B_err", "corr_AB", "rms_resid", "r2", "quantile"]]

,A,A_err,A_lo,A_hi,B,B_err,corr_AB,rms_resid,r2,quantile
legacy (normal z),5.832058,0.035868,5.761758,5.902358,67.11909,9.246427,0.808392,0.042227,0.929443,1.959964
staging (t),5.9084,0.059151,5.74417,6.072631,82.048757,15.248655,0.808392,0.069638,0.878612,2.776445
"staging, root recipe (t)",6.158758,0.455885,4.893018,7.424498,190.457791,117.522823,0.808392,0.536708,0.39635,2.776445


### Legacy Fig 6 (as published: 18-point Tian CSV, raw and smoothed 1D curves, 3390 km cell width)

In [12]:
tian_csv = np.loadtxt(LEGACY_ZENODO / "tian1d_dataset.csv", delimiter=",")
sph_csv = np.loadtxt(LEGACY_ZENODO / "spherical_1d_data.csv", delimiter=",")
refs_legacy = [
    dict(x=tian_csv[:, 0], y=tian_csv[:, 1], label="Tian et al., 2023", color="k"),
    dict(x=sph_csv[:, 0], y=sph_csv[:, 1], label="Spherical 1D", color="#ff7f0e"),
    dict(x=savgol_filter(p26_growth, window_length=5, polyorder=2), y=p26_alt, label="Harrah 1D", color="#2ca02c"),
]
fig = plotting.plot_growth_profile_facets(R.add_distance_from_centre(legacy, LEGACY_RADIUS_KM), refs_legacy)
fig.savefig(FIG_DIR / "growth_profiles_S1-S6_legacy.png", dpi=300, bbox_inches="tight")